# **Import libraries and establish filepaths**

In [1]:
# Import libraries
import pandas as pd

# Load dataframe
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

from config.project_paths import (
    ROOT,
    DATA_RAW,
    DATA_PROCESSED,
    FIGURES,
    YEAR_MIN,
    YEAR_MAX,
    get_raw_data_path,
    get_processed_data_path,
    get_figure_path,
    validate_paths,
)


FILE = get_raw_data_path("Table-7.3-Number-of-registered-jobs-with-SVB-by-economic-sector-by-gender-2015-2024.xlsx")

# Test for errors
FILE

PosixPath('/home/ggirelli/Documents/DataAnalysis/Projects/Capstone_1/data/raw/Table-7.3-Number-of-registered-jobs-with-SVB-by-economic-sector-by-gender-2015-2024.xlsx')

---
**Load and inspect DataFrame**
---

In [23]:
# Main dataframe
df_main = pd.read_excel(FILE, header=None)

# df_main.columns
# df_main.head()
# df_main.info()

df_main.iloc[3:, :2]

df_filtered = df_main.copy()

df_filtered = df_filtered.reset_index(drop=True)

valid_rows = (
    df_filtered[1].notna()
) & (
    ~df_filtered[1].str.contains("Economic")
) & (
    ~df_filtered[1].str.contains("Total")
) & (
    ~df_filtered[1].str.contains("Source")
)

df_filtered = df_filtered[valid_rows]

df_filtered[1] = df_filtered[1].str.replace("\t", "")

df_filtered[1].unique()

df_filtered[0].isna().sum()

df_filtered[df_filtered[0].notna()].copy()

NameError: name 'columns' is not defined

# **Create and combine columns**

In [3]:
# Create columns
sex_row = df_main.iloc[1]
year_row = df_main.iloc[2]

# Use forward fill to fill in NaNs
sex_row =  sex_row.ffill()

# Convert years to numeric value
year_row_cleaned = pd.to_numeric(year_row, errors="coerce")

#
sex_row = df_main.iloc[1, 2:].ffill()
year_row_cleaned = pd.to_numeric(df_main.iloc[2, 2:], errors="coerce")

In [4]:
# # Validity mask / this keeps only columns where a year exists
# boolean mask
valid_cols = year_row_cleaned.notna()

# Valid cols test
valid_cols
valid_cols.index

# Apply validity mask to both cols
sex_row =  sex_row[valid_cols]
year_row_cleaned = year_row_cleaned[valid_cols]

# Combine using zip()
cleaned_columns = list(zip(year_row_cleaned.astype(int), sex_row))

# Inspect column with sector names
data = df_main.iloc[3:, :].copy()

#data = data.loc[:, valid_cols.index[valid_cols]]
print(data)

                              0   \
3                          A , B   
4                             C    
5                             D    
6                             E    
7                             F    
8                            NaN   
9                            NaN   
10                           NaN   
11                            G    
12                            H    
13                            I    
14                            J    
15                            K    
16                             L   
17                            M    
18                            N    
19                            O    
20                             P   
21                             Q   
22                             R   
23                             S   
24                           NaN   
25                           NaN   
26                           NaN   
27                           NaN   
28  Source: Social Security Bank   

                           

---
# **Transform data from wide to long format**

In [5]:
#data = data.set_index("Economic sector")